# Prepare the data

Cuts the three scenes into tiles and burns my outlines into label rasters.

Each scene gets tiled in its own coordinate system. I resample to 0.125 m/px but
do not warp anything onto a shared grid, so the labels sit exactly where I drew
them.

Per tile three files come out:

```
<scene>_<col>_<row>.tif            image, uint16, raw values
<scene>_<col>_<row>_masks.tif      one integer per boulder
<scene>_<col>_<row>_corridor.tif   1 where I actually looked
index.json                         bounds, fold and counts per tile
```

Runs on the laptop, the source scenes are here. Only the finished archive goes
to the server.

In [ ]:
import sys, json
import numpy as np
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
import config
from common import tiles, folds

config.check_paths()        # clear error if the data is somewhere else
config.make_dirs()
print("raw     ", config.RAW)
print("archive ", config.ARCHIVE)
print("res     ", config.RES, "m/px |  tile", config.TILE, "px =",
      config.TILE * config.RES, "m |  overlap", config.OVERLAP, "px")

## What we are starting from

In [ ]:
rows = []for s, info in config.SCENES.items():    p = config.image(s)    with rasterio.open(p) as r:        rows.append((s, info["date"], p.name, r.count, f"{r.res[0]:.4f}",                     f"{r.width} x {r.height}", str(r.dtypes[0]),                     f"{config.RES / r.res[0]:.2f}x"))print(f"{'scene':6}{'date':12}{'file':28}{'b':>2} {'m/px':>8} {'size':>16} {'type':>8} {'upsample':>9}")for r in rows:    print(f"{r[0]:6}{r[1]:12}{r[2]:28}{r[3]:>2} {r[4]:>8} {r[5]:>16} {r[6]:>8} {r[7]:>9}")

## Outlines

Sanity check that the files are the ones I think they are.

In [ ]:
out = {}for s in config.SCENES:    g = gpd.read_file(config.OUTLINES / f"{s}_bloecke.gpkg").to_crs(config.CRS)    g["d_m"] = 2 * np.sqrt(g.area / np.pi)    out[s] = g    ok = g.d_m >= config.MIN_DIAMETER    tr = g.d_m >= config.MIN_TRAINING    print(f"{s}: {len(g):5d} total | {tr.sum():5d} >= {config.MIN_TRAINING} m (training)"          f" | {ok.sum():5d} >= {config.MIN_DIAMETER} m (evaluated)"          f" | D50 {g.d_m[ok].median():.2f} m | max {g.d_m.max():.1f} m")

In [ ]:
## Overlapping outlines

A label raster has one integer per pixel, so two outlines that overlap cannot
both survive, the second one overwrites the first. Worth knowing how often that
happens before it silently eats boulders.

## Overlapping polygonsA label raster holds one integer per pixel, so overlapping outlines cannot berepresented — the later one would overwrite the earlier. Check how bad it is.

In [ ]:
## Folds

Cuts are absolute kilometres along the valley, same in every scene. The two
ends are always training because one scene is missing there (february starts at
km 7.7, march stops at 24.7).

## Fold assignment

In [ ]:
## Build the archive

A few minutes per scene. Each one reads and resamples roughly 150 Mpx.

## Build the archiveTakes a few minutes per scene — it reads and resamples roughly 150 Mpx each.

In [ ]:
## What came out

## Archive overview

In [ ]:
## Look at a few tiles

Red outlines are the labels. If these look wrong, something upstream is wrong.

## Visual check

In [ ]:
## Copy it over

```bash
rsync -av --progress   "/c/Users/pauls/Desktop/Potsdam_Master/Internship_Uni/trishuli_netz_data/archive/"   pcpool:/DATA/schrepel/trishuli_daten/archive/
```

## Send to the server```bashrsync -av --progress \  "/c/Users/pauls/Desktop/Potsdam_Master/Internship_Uni/trishuli_netz_data/archive/" \  pcpool:/DATA/schrepel/trishuli_daten/archive/```